# Tarea Práctica: Riesgos de Integridad en Entornos Distribuidos

## Contexto
Trabajas como Analista de Datos en un *E-commerce* de rápido crecimiento. El sistema de procesamiento de datos es distribuido: eventos de navegación y transacciones llegan continuamente desde microservicios separados (inventario, carrito, pasarela de pagos) a través de un sistema de mensajería (como Kafka).

Al ser una arquitectura distribuida (con retardos de red, reintentos automáticos y equipos de desarrollo separados), empiezan a aparecer comportamientos anómalos en los registros analíticos diarios.

## Objetivo
Tu objetivo no es la simple "limpieza por valores atípicos" (como hicimos en IoT), sino reparar **riesgos críticos de integridad técnica y lógica** en un *subset* representativo de los datos.

Debes abordar los siguientes problemas extraídos de la **UT3 - Capítulo 2**:
1.  **Duplicados y Replays:** El broker de mensajería reintenta envíos si falla la red, inyectando transacciones repetidas.
2.  **Estados Parciales:** Fallos en el pipeline de enriquecimiento hacen que algunas compras lleguen sin asociar al usuario (`user_id`).
3.  **Corrupción Lógica Silenciosa:** Un cambio silencioso de esquema en origen provoca que números lleguen como textos con formato europeo (ej: `"14,50 €"`), rompiendo los cálculos y devolviendo `NaN` tras conversiones fallidas.

### ¡IMPORTANTE! ⚠️
Como siempre, **justifica** por qué tomas cada decisión (¿borras un nulo? ¿rellenas? ¿te fías de qué columna para borrar duplicados?). Si delegas la justificación exclusivamente a una IA y no refleja tu entendimiento de la teoría, se notará y mas cuando te pregunte en clase.🥸 No me seas, y se curioso y pierde un poco el tiempo en amprender. 


In [1]:
import pandas as pd
import numpy as np

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---
# Tienes una muestra de 8 eventos de la última hora en la Capa Bronce (Raw).

data = {
    'event_id': ['EV-100', 'EV-101', 'EV-101', 'EV-102', 'EV-103', 'EV-104', 'EV-105', 'EV-106'],
    'timestamp': [
        '2026-03-01 14:00:00', 
        '2026-03-01 14:05:00', 
        '2026-03-01 14:05:00',  # Reintento exacto del EV-101
        '2026-03-01 14:10:00',
        '2026-03-01 14:12:00',
        '2026-03-01 14:15:00',
        '2026-03-01 14:10:00',  # Evento rezagado (old timestamp)
        '2026-03-01 14:20:00'
    ],
    'user_id': ['U-01', 'U-02', 'U-02', None, 'U-03', 'U-01', 'U-04', 'U-05'], # El EV-102 perdió la identidad del usuario
    'action': ['view', 'purchase', 'purchase', 'purchase', 'view', 'purchase', 'view', 'purchase'],
    'amount': [0.0, 50.5, 50.5, 120.0, 0.0, "25,99 €", 0.0, -10.0] # Falla esquema y valor negativo
}

df_events = pd.DataFrame(data)
print("--- Dataset Raw (Sucio) extraído de Data Lake ---")
display(df_events)

--- Dataset Raw (Sucio) extraído de Data Lake ---


,event_id,timestamp,user_id,action,amount
0,EV-100,2026-03-01 14:00:00,U-01,view,0.0
1,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
2,EV-101,2026-03-01 14:05:00,U-02,purchase,50.5
3,EV-102,2026-03-01 14:10:00,None,purchase,120.0
4,EV-103,2026-03-01 14:12:00,U-03,view,0.0
5,EV-104,2026-03-01 14:15:00,U-01,purchase,"25,99 €"
6,EV-105,2026-03-01 14:10:00,U-04,view,0.0
7,EV-106,2026-03-01 14:20:00,U-05,purchase,-10.0


---
### Ejercicio 1: Idempotencia y Duplicados por Reintentos (Replays)
*(Ref: UT3 punto 2.1c)*

**Problema:** En sistemas distribuidos, si el consumidor no confirma haber recibido la transacción `EV-101`, el productor la reenvía *"por si acaso"*. Si sumamos directamente los ingresos, estaríamos contando más dinero del real.

**Tarea:** 
1. Detecta qué fila está duplicada, garantizando la idempotencia basándote exclusivamente en el identificador único de la transacción (`event_id`).
2. Elimina la fila fantasma manteniendo de forma segura la original.

In [ ]:
# Cuando el broker (Kafka, RabbitMQ...) no recibe el ACK del consumidor,
# reenvia el mensaje por si acaso. El EV-101 aparece dos veces por eso.
# Si sumamos amounts directamente contamos 50.5 de mas -> contabilidad mal.
#
# Deuplico SOLO por event_id porque es el identificador unico de la transaccion.
# No uso timestamp ni amount porque si cambiaran minmamente en el reenvio
# no lo detectaria. Dos filas con mismo event_id = misma operacion seguro.
duplicados = df_events.duplicated(subset=["event_id"], keep="first")
print(f"Replays detectados: {duplicados.sum()}")
display(df_events[duplicados])

df_events = df_events.drop_duplicates(subset=["event_id"], keep="first")
print("\nDataset sin replays:")
display(df_events)

---
### Ejercicio 2: Estados Parciales y Registros Huérfanos
*(Ref: UT3 punto 2.1e)*

**Problema:** El evento `EV-102` es una compra real (`amount = 120.0`) pero se ha quedado en **estado parcial**. Existe, pero le falta el `user_id` asociado porque un microservicio que enriquece los datos de cliente estaba caído.

**Tarea:**
1. A diferencia del caso del "Edificio Inteligente" donde la falta de temperatura se podía interpolar, imputar un `user_id` a la ligera inventándolo puede arruinar métricas críticas como el Ticket Medio por Usuario o cruces de facturación.
2. Demuestra cómo identificarías las compras "huérfanas" (`action == 'purchase'` sin `user_id`).
3. Elimina o mueve ese registro a un dataframe de "cuarentena" justificando tu elección.

In [ ]:
# Esto NO es como el caso IoT donde faltaba una temperatura y la rellenabamos
# con la media. Aqui falta un user_id, que es un dato de negocio. No me lo
# puedo inventar: si asigno la compra de 120 euros a un usuario cualquiera,
# falseo metricas como el Ticket Medio o el cruce de facturacion.
#
# Lo mando a cuarentena: lo aparto para investigar despues (quizas el micro
# de enriquecimiento se recupere y podamos completarlo). No lo borro porque
# la compra SI ocurrio, solo falta saber quien la hizo.
compras_huerfanas = df_events[(df_events["action"] == "purchase") & (df_events["user_id"].isnull())]
print(f"Compras huerfanas: {len(compras_huerfanas)}")
display(compras_huerfanas)

df_cuarentena = compras_huerfanas.copy()
df_events = df_events[~((df_events["action"] == "purchase") & (df_events["user_id"].isnull()))]

print("\nDataset sin huerfanos:")
display(df_events)
print("\nCuarentena:")
display(df_cuarentena)

---
### Ejercicio 3: Cambio de Esquema y Corrupción Silenciosa Lógica
*(Ref: UT3 punto 2.2c y 2.1b)*

**Problema:** Uno de los equipos front-end hizo una actualización silenciosa y empezó a enviar los importes concatenados con texto español (`"25,99 €"`) en la compra `EV-104`.
Al mezclarse números y Strings `amount` pasa a ser una columna `Object`. También hay un valor sin sentido (`-10.0`).

**Tarea:**
1. Muestra el tipo de dato (`dtypes`) actual de la columna `amount`.
2. Repara la columna: limpia el texto (cambia la coma por punto y quita el " €") solo en las celdas que sean texto, o fuerza a que toda la columna se convierta a `float`. `pd.to_numeric(errors='coerce')` puede ser tu gran aliado táctico.
3. Las compras no pueden tener costes negativos. Localiza el valor numérico contaminado y arréglalo pasándolo a 0 o borrando la fila.

In [ ]:
# Corrupcion silenciosa: el front-end actualizo y empezo a mandar "25,99 €"
# como texto en vez del numero 25.99. Al mezclar tipos, Pandas pone la
# columna como Object y cualquier sum() o mean() da mal.
print("Tipo actual de amount:", df_events["amount"].dtype)

# Limpio el texto: quito el euro, cambio coma por punto
def limpiar_amount(valor):
    if isinstance(valor, str):
        valor = valor.replace("€", "").replace(",", ".").strip()
    return valor

df_events["amount"] = df_events["amount"].apply(limpiar_amount)

# Fuerzo a float. El errors="coerce" es clave: si algo no se puede convertir
# lo pone NaN en vez de reventar el pipeline entero
df_events["amount"] = pd.to_numeric(df_events["amount"], errors="coerce")
print("Tipo reparado:", df_events["amount"].dtype)

# Compras negativas no tienen sentido (seria una devolucion, otro tipo de evento)
df_events.loc[df_events["amount"] < 0, "amount"] = 0.0

print("\nDataset reparado:")
display(df_events)

---
### Resultado Final Consolidado
Prueba a sumar todos los `amount` de la columna de las compras resultantes. Tu contabilidad debería ser perfecta ahora.

In [ ]:
compras = df_events[df_events["action"] == "purchase"]
print("=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
display(compras[["event_id", "user_id", "action", "amount"]])
print(f"\nTotal ingresos: {compras['amount'].sum():.2f} euros")